# Train RandLA-Net on DALES Aerial LiDAR Dataset

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/opengeos/geoai/blob/main/docs/examples/train_point_cloud_dales.ipynb)

This notebook trains a RandLA-Net model on the [DALES](https://udayton.edu/engineering/research/centers/vision_lab/research/was_702702702702702702/dales.php)
aerial LiDAR dataset (500M+ points, 10 km^2, 9 classes) and uploads
the checkpoint to Hugging Face Hub.

**Requirements:** GPU runtime on Google Colab (T4 or better).

**Classes:** unknown, ground, vegetation, cars, trucks, power_lines, fences, poles, buildings

## 1. Install dependencies

In [ ]:
# Colab: install compatible PyTorch + geoai + Open3D
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu121
!pip install "geoai-py[pointcloud]" "numpy<2" huggingface_hub

## 2. Download and prepare DALES

DALES provides `.las` files with ground-truth classification labels.
Download from the [official site](https://udayton.edu/engineering/research/centers/vision_lab/research/was_702702702702702702/dales.php)
and upload to your Google Drive, or use `gdown` if you have a direct link.

In [ ]:
import os
from pathlib import Path

# Mount Google Drive (store large dataset there)
from google.colab import drive
drive.mount("/content/drive")

# Set paths - adjust these to where you stored DALES
DALES_ROOT = "/content/drive/MyDrive/datasets/DALES"
TRAIN_DIR = os.path.join(DALES_ROOT, "train")
VAL_DIR = os.path.join(DALES_ROOT, "test")  # DALES uses test split for val
SAVE_DIR = "/content/checkpoints"

os.makedirs(SAVE_DIR, exist_ok=True)

train_files = sorted(Path(TRAIN_DIR).glob("*.las"))
val_files = sorted(Path(VAL_DIR).glob("*.las"))
print(f"Train files: {len(train_files)}, Val files: {len(val_files)}")

## 3. Initialize model and train

In [ ]:
from geoai.pointcloud import PointCloudClassifier

# Initialize with DALES config (downloads SemanticKITTI weights as
# starting point for transfer learning, architecture is overridden
# by DALES config).
classifier = PointCloudClassifier(
    model_name="RandLANet_DALES",
    device="cuda",
)

In [ ]:
# Train - this will take several hours on a T4 GPU
history = classifier.train(
    train_dir=TRAIN_DIR,
    val_dir=VAL_DIR,
    epochs=100,
    learning_rate=0.001,
    batch_size=4,
    save_dir=SAVE_DIR,
)
print(f"Checkpoint saved: {history['checkpoint_path']}")

## 4. Test inference on a sample file

In [ ]:
import numpy as np

# Pick a validation file to test
test_file = str(val_files[0])
output_file = "/content/test_classified.las"

labels, probs = classifier.classify(test_file, output_path=output_file)
print(f"Classified {len(labels)} points into {len(np.unique(labels))} classes")

stats = classifier.summary(output_file)
print(f"\nClass distribution:")
for name, pct in stats["class_percentages"].items():
    count = stats["class_counts"][name]
    print(f"  {name:20s}: {count:>10,} ({pct:.1f}%)")

## 5. Upload checkpoint to Hugging Face Hub

Upload the trained checkpoint so it can be downloaded by `geoai`.
You'll need a [Hugging Face token](https://huggingface.co/settings/tokens).

In [ ]:
import hashlib

from huggingface_hub import HfApi, login

# Authenticate (paste your HF token when prompted)
login()

checkpoint_path = history["checkpoint_path"]

# Compute SHA-256 for integrity verification
with open(checkpoint_path, "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()
print(f"SHA-256: {sha256}")
print(f"Update SUPPORTED_MODELS['RandLANet_DALES']['sha256'] with this value.")

# Upload to Hugging Face
api = HfApi()
api.create_repo("jayakumarpujar/randlanet-dales", repo_type="model", exist_ok=True)
api.upload_file(
    path_or_fileobj=checkpoint_path,
    path_in_repo="randlanet_dales.pth",
    repo_id="jayakumarpujar/randlanet-dales",
)
print("Checkpoint uploaded to: https://huggingface.co/jayakumarpujar/randlanet-dales")

## 6. Next steps

After uploading the checkpoint:

1. Copy the SHA-256 hash printed above
2. Update `SUPPORTED_MODELS["RandLANet_DALES"]["sha256"]` in `geoai/pointcloud.py`
3. Push the update to the PR branch
4. Users can now run:

```python
from geoai.pointcloud import classify_point_cloud

labels, probs = classify_point_cloud(
    "my_aerial_lidar.las",
    output_path="classified.las",
)
```